In [5]:
from openai import OpenAI
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

VECTOR_STORE_NAME = "rule_demo"

vector_store = client.vector_stores.create(
    name=VECTOR_STORE_NAME,
    # 필요하면 metadata, description 등도 추가 가능
)

print(vector_store.id)


vs_6a7e70f5d54081918aa389bfa2166faf


In [6]:
chunks = [
    {
        "id": "doc1-chunk0",
        "content": "결제 수단은 계정 설정의 결제 정보에서 변경할 수 있습니다...",
        "metadata": {"doc_id": "doc1", "section": "billing", "lang": "ko"},
    },
    # ...
]

for chunk in chunks:
    client.vector_stores.items.create(
        vector_store_id=vector_store.id,
        input=chunk["content"],
        metadata=chunk["metadata"],
        id=chunk["id"],
        # embedding은 내부에서 자동 생성되도록 맡기거나
        # 미리 embedding 모델을 지정할 수도 있음
    )

AttributeError: 'VectorStores' object has no attribute 'items'

In [3]:
results = client.vector_stores.query(
        vector_store_id=vector_store.id,
        input=query,
        top_k=k,
        # 필요하면 filter에 metadata 조건을 줄 수도 있다 (예: lang == 'ko')
    )

AttributeError: 'VectorStores' object has no attribute 'query'

In [7]:
# Vector Store 생성
vector_store = client.vector_stores.create(
  name="knou_rules",
  metadata={
    "description": "한국방송통신대학교 학칙"
   },
  chunking_strategy={
    'type': 'static',
    'static': {
      'max_chunk_size_tokens': 800,
      'chunk_overlap_tokens': 400
    }
  } # 또는 auto
)

In [8]:
vector_store_list = client.vector_stores.list()
print(vector_store_list)

SyncCursorPage[VectorStore](data=[VectorStore(id='vs_6a7e719e2274819188997b3193f5e62a', created_at=1786671518, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1786671518, metadata={'description': '한국방송통신대학교 학칙'}, name='knou_rules', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None), VectorStore(id='vs_6a7e70f5d54081918aa389bfa2166faf', created_at=1786671350, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1786671350, metadata={}, name='rule_demo', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None), VectorStore(id='vs_6a7e6f1f86ec8191a47937bca4f70975', created_at=1786670879, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1786670879, metadata={}, name='rule_demo', object='vector_store', status='completed', usage_by

In [12]:
# 파일 업로드
file = client.files.create(
    file=open("/content/sample_data/한국방송통신대학교 학칙.pdf", "rb"),
    purpose="assistants"
)

# Vector Store에 파일 추가
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file.id
)

VectorStoreFile(id='file-Wrjay3GVWe94RCcm5McMYg', created_at=1786671970, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_6a7e719e2274819188997b3193f5e62a', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [14]:
query = "방송통신대에는 어떤 학과가 있나요?"

# 벡터스토어 검색 수행
search_results = client.vector_stores.search(
  vector_store_id=vector_store.id,
  query=query,
  max_num_results=5
)

for result in search_results.data:
  print(result.score, result.content)

print(search_results.data[0].content[0].text)



0.7758716434912505 [Content(text='제4절 부속 시설 등\r\n제22조(부속시설 등) ① 시행령 제6조에 따라 지역대학, 디지털미디어센터를 둔다.\r\n② 「대학도서관진흥법」 제6조 제1항에 따라 본교에 교육기본시설로 중앙도서관을 둔다.\r\n③ 본교에 다음 각 호의 부속시설을 둔다.\r\n1. 평생교육원\r\n2. 종합교육연수원\r\n3. 교양교육원\r\n4. 역사기록관\r\n5. 삭제(2024.1.19.)\r\n6. 국제협력단\r\n7. 산학협력단\r\n8. 인권센터(신설)\r\n9. 삭제(2026.4.8.)\r\n10. 교원양성지원센터(신설 2024.1.19.)\r\n④ 본교에 다음 각 호의 연구시설을 둔다.\r\n1. 원격교육혁신연구원 (개정 2022.12.20., 2026.4.8.)\r\n2. 통합인문학연구소\r\n⑤ 각 부속시설 등의 장은 관련 단과대학, 학과(부)의 장이 겸직하되, 부속시설 등의 특수성을 반영하여 총장이\r\n별도로 임명할 수 있다.\r\n⑥ 총장은 「대학 설립·운영 규정」 제4조제1항에 따른 연구시설에 대해서는 2년마다 해당시설의 운영 실적을 평\r\n가하여 존속 또는 폐지 여부를 결정한다. 이 경우 평가에 관한 세부 사항은 총장이 정한다.\r\n제23조(지역대학) ① 입학, 수업, 시험, 장학의 학업 지원과 학생 활동을 지원하고 지역사회의 평생교육 진흥에 기여\r\n하기 위하여 지역대학을 둔다.법제처 4 국가법령정보센터\r\n한국방송통신대학교 학칙\r\n② 지역대학에 학장을 두며, 학장은 소관 지역대학의 업무를 총괄하고 소속직원을 지휘·통솔한다.', type='text')]
0.7690125241831166 [Content(text='5. 부속시설과 부속기관·각종 위원회의 설치·폐지 및 운영에 관한 사항\r\n6. 그 밖에 총장이 회의에 올리는 사항\r\n⑥ 교무위원회의 운영에 관한 세부 사항은 총장이 따로 정한다.\r\n제32조(학·처장회의) ① 단과대학 운영 및 프라임칼리지 학사학위과정 운영에 관한 

In [17]:
question = "방송통신대에는 어떤 학과가 있나요?"

response = client.responses.create(
    model="gpt-4.1-mini",
    input=question,
    instructions="당신은 학사정보 상담사입니다. 제공된 문서를 기반으로 정확한 답변을 제공해주세요.",
    temperature=0.2,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
            "max_num_results": 10
        }
    ],
    include=["file_search_call.results"]

)

print(response.output_text)
print(response.output[0].results[0].text)

한국방송통신대학교(방송통신대)의 학과(부) 구성에 대한 구체적인 학과명은 제공된 학칙 문서 내에 직접적으로 나열되어 있지 않습니다. 다만, 학칙에 따르면 학과(부)는 단과대학이나 프라임칼리지, 대학원 내에 설치되며, 각 학과(부)에는 학과(부)장이 있고, 전공주임교수가 전공별로 임명되어 학과 운영과 학생 교육을 담당합니다.

또한, 계약학과(부)를 국가, 지방자치단체 또는 산업체 등과의 계약에 의해 설치·운영할 수 있다고 명시되어 있습니다(제8조). 그리고 부속시설로는 평생교육원, 종합교육연수원, 교양교육원, 역사기록관, 국제협력단, 산학협력단, 인권센터, 교원양성지원센터 등이 있습니다.

학과별 구체적인 명칭이나 목록은 학칙 문서에 포함되어 있지 않으므로, 방송통신대 공식 홈페이지나 별도의 학과 안내 자료를 참고하시는 것이 정확합니다.

요약하면, 방송통신대는 여러 단과대학과 프라임칼리지, 대학원 내에 학과(부)를 두고 있으며, 계약학과도 설치할 수 있습니다. 구체적인 학과명은 학칙 문서에 명시되어 있지 않습니다. 필요시 추가 자료를 요청해 주세요.
④ 총장은 소관 사무 중 일부를 학(원)장에게 위임할 수 있다.
⑤ 학과(부)장은 학과(부)의 운영을 통할하며, 소속 전임교수 및 조교를 통솔하고, 학생의 교육과 지도에 관하여
학(원)장을 보좌한다.
제10조(전공주임교수) ① 제6조 및 제7조에서 정하는 학과(부)·전공에는 다음 각 호와 같이 전공주임교수를 둘 수 있
다.
1. 전공 분리학과(학부 포함)
2. 연계전공
3. 경영대학원 각 전공
② 전공주임교수는 전임교수 중에서 학과(부)장(단, 연계전공은 주관학과(부) 학과(부)장) 및 경영대학원장의 추천
에 따라 총장이 임명하며, 전공의 운영 및 학생 교육과 지도에 관하여 학부장 또는 학과장을 보좌한다.
③ 전공주임교수 운영에 관한 세부 사항은 총장이 따로 정한다.
제11조(전임교수의 소속 및 교수시간) ① 본교의 전임교수는 원칙적으로 그 전공에 따라 단과대학·프라임칼리지 또
는 대학원의 1개 학과(부)에 소속